# MIT–BIT fixed energy budget controller

Compare fixed 1C, the model-guided moving-target controller, and a 15-cycle fixed-budget controller. The new controller forecasts energy, plans over the remaining schedule, and commits one five-cycle action at a time.

Run all cells from the LifeGuard-RL project with the environment used by Notebook 17. This experiment reads the Notebook 14, 16, and 17 reports as folders or ZIPs. It reuses their audited outcomes and runs on one CPU thread.

In [1]:
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[name] = "1"

import json
from collections import defaultdict
from dataclasses import dataclass
from itertools import islice, product
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits

NOTEBOOK_14_EXPORT = None
NOTEBOOK_16_EXPORT = None
NOTEBOOK_17_EXPORT = None
OUTPUT_ROOT = None

BLOCK = 5
BLOCKS = 3
HORIZONS = tuple(range(1, BLOCKS + 1))
RATES = (1, 2, 3)
FOLDS = (1, 2, 3, 4, 5)
POLICY_STARTS = (11, 41, 66)
FIRST_START, LAST_START = 11, 91
ENERGY_TARGET = 0.90
OVERPREDICTION_QUANTILE = 0.90
TOLERANCE = 1e-9
BOOTSTRAP_DRAWS = 2000
SEED = 42
KEYS = ["cell_id", "start_cycle"]
POLICIES = {"fixed_1c": "Fixed 1C", "model": "Moving-target controller",
            "energy_budget": "Fixed-budget controller"}
OUTPUT_NAME = "lifeguard_mit_bit_energy_budget_controller_results"

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents)
                if (p / "README_LifeGuard_RL.md").is_file() and (p / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")



def report_path(override, name):
    if override is not None:
        return Path(override).expanduser()
    path = project / "reports" / name
    return path if path.is_dir() else path.with_suffix(".zip")


def read_tables(path, names):
    if path.is_dir():
        return {name: pd.read_csv(path / f"{name}.csv") for name in names}
    if not path.is_file():
        raise FileNotFoundError(f"Missing prerequisite report: {path}")
    with ZipFile(path) as bundle:
        tables = {}
        for name in names:
            files = [file for file in bundle.namelist() if PurePosixPath(file).name == f"{name}.csv"]
            if len(files) != 1:
                raise ValueError(f"Expected one {name}.csv in {path}.")
            with bundle.open(files[0]) as stream:
                tables[name] = pd.read_csv(stream)
        return tables


def require_unique(table, keys, label):
    if table[keys].isna().any().any() or table.duplicated(keys).any():
        raise ValueError(f"Missing or duplicate {keys} in {label}.")


source14 = report_path(NOTEBOOK_14_EXPORT, "lifeguard_mit_bit_switch_replay_results")
source16 = report_path(NOTEBOOK_16_EXPORT, "lifeguard_mit_bit_controller_benchmark_results")
source17 = report_path(NOTEBOOK_17_EXPORT, "lifeguard_mit_bit_multiblock_simulator_results")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else project / "reports") / OUTPUT_NAME

In [2]:
@dataclass(frozen=True)
class Block:
    start: int
    rate: int
    ah: float
    wh: float
    first_ah: float
    last_ah: float
    minutes: float


def build_trace(block_audit, cycle_audit, timing):
    require_unique(block_audit, KEYS, "Notebook 14 blocks")
    require_unique(cycle_audit, ["cell_id", "aging_cycle"], "Notebook 14 cycles")
    require_unique(timing, ["cell_id", "aging_cycle"], "Notebook 16 cycle times")
    time_blocks = timing.groupby(KEYS).charge_minutes.sum(min_count=BLOCK)
    raw = {int(cell): group.set_index("aging_cycle").sort_index()
           for cell, group in cycle_audit.groupby("cell_id")}
    traces = defaultdict(dict)
    for row in block_audit.itertuples(index=False):
        if row.status != "ok" or row.charge_rate_c not in RATES:
            continue
        start = int(row.start_cycle)
        part = raw[row.cell_id].reindex(range(start, start + BLOCK))
        capacities = part.capacity_ah.to_numpy(dtype=float)
        if np.isfinite(capacities).all():
            minutes = time_blocks.get((row.cell_id, start), np.nan)
            traces[row.cell_id][start] = Block(start, int(row.charge_rate_c), float(np.median(capacities)),
                                              float(row.observed_wh), capacities[0], capacities[-1], float(minutes))
    return dict(traces)


def initial_history(trace, start, require_energy=True):
    if start - BLOCK not in trace or start - 2 * BLOCK not in trace:
        return None
    history = tuple(trace[cycle] for cycle in sorted(trace) if cycle < start)
    if require_energy and (not np.isfinite(history[-1].wh) or history[-1].wh <= 0):
        return None
    return history


def feature_row(history):
    previous, latest = history[-2:]
    start = latest.start + BLOCK
    result = {
        "start_ah": latest.ah, "latest_ah": latest.last_ah,
        "recent_block_delta_ah": latest.ah - previous.ah,
        "within_block_delta_ah": latest.last_ah - latest.first_ah,
        "age_cycles": start - 1, "first_rate_c": history[0].rate,
        "previous_rate_c": latest.rate,
        "prior_2c_blocks": sum(block.rate == 2 for block in history),
        "prior_3c_blocks": sum(block.rate == 3 for block in history),
    }
    for rate in RATES:
        same = [block for block in history if block.rate == rate][-3:]
        last = same[-1].ah if same else np.nan
        gap = start - same[-1].start if same else np.nan
        slope = (last - same[0].ah) / (same[-1].start - same[0].start) if len(same) > 1 else 0
        result[f"last_{rate}c_ah"] = last
        result[f"lag_{rate}c_cycles"] = gap
        result[f"projected_{rate}c_ah"] = min(last, last + gap * slope) if same else np.nan
    return result


def prepare_examples(events, traces):
    origins, features, labels = [], [], []
    for row in events.itertuples(index=False):
        trace = traces[row.cell_id]
        history = initial_history(trace, row.start_cycle, require_energy=False)
        if history is None:
            continue
        block = trace.get(row.start_cycle)
        origins.append((int(row.cell_id), int(row.start_cycle), int(row.fold), int(row.charge_rate_c)))
        features.append(feature_row(history))
        if block is None:
            labels.append((np.nan,) * 5 + (history[-1].wh,))
            continue
        if row.charge_rate_c != block.rate:
            raise ValueError(f"Recorded rate disagreement for cell {row.cell_id}, cycle {row.start_cycle}.")
        np.testing.assert_allclose(row.observed_wh, block.wh, atol=1e-9, rtol=0)
        labels.append((block.ah, block.wh, block.first_ah - block.ah,
                       block.last_ah - block.ah, block.minutes, history[-1].wh))
    return (pd.DataFrame(origins, columns=[*KEYS, "fold", "charge_rate_c"]),
            pd.DataFrame(features),
            pd.DataFrame(labels, columns=["ah", "wh", "first_offset_ah", "last_offset_ah", "minutes", "previous_wh"]))


def recorded_windows(traces, origins, cohort):
    windows = []
    for cell, start, model_fold, rate in origins[[*KEYS, "fold", "charge_rate_c"]].itertuples(index=False, name=None):
        if not FIRST_START <= start <= LAST_START:
            continue
        trace = traces[cell]
        history = initial_history(trace, start)
        if history is None:
            continue
        recorded = []
        for step in range(max(HORIZONS)):
            cycle = start + step * BLOCK
            block = trace.get(cycle)
            if cycle > LAST_START or block is None or not np.isfinite([block.ah, block.wh, block.minutes]).all():
                break
            recorded.append(block)
        if recorded:
            if recorded[0].rate != rate:
                raise ValueError("Recorded window rate disagrees with the audited assignment.")
            windows.append(dict(cohort=cohort, cell_id=cell, start_cycle=start, fold=model_fold,
                                history=history, path=tuple(recorded)))
    return windows


def validate_folds(events, splits):
    require_unique(events, KEYS, "Notebook 14 predictions")
    require_unique(splits, ["fold", "cell_id"], "Notebook 14 cell splits")
    if set(splits.fold) != set(range(1, 6)) or events.cell_id.nunique() != 55:
        raise ValueError("Notebook 15 expects the full 55-cell, five-fold Notebook 14 benchmark.")
    test_fold = splits.loc[splits.role.eq("test")].set_index("cell_id", verify_integrity=True).fold
    cells = set(events.cell_id)
    if set(test_fold.index) != cells or not events.cell_id.map(test_fold).eq(events.fold).all():
        raise ValueError("Notebook 14 prediction rows and held-out folds disagree.")
    expected = {(fold, cell, "test" if test_fold[cell] == fold else "train")
                for fold in range(1, 6) for cell in cells}
    if set(splits[["fold", "cell_id", "role"]].itertuples(index=False, name=None)) != expected:
        raise ValueError("The saved train/test split is incomplete.")


audit14 = read_tables(source14, ("predictions", "cell_splits", "block_audit", "cycle_audit"))
audit16 = read_tables(source16, ("charge_time_audit", "calibration"))
audit17 = read_tables(source17, ("policy_trajectories", "recorded_rollout_totals"))
events = audit14["predictions"].sort_values(KEYS).reset_index(drop=True)
validate_folds(events, audit14["cell_splits"])
traces = build_trace(audit14["block_audit"], audit14["cycle_audit"], audit16["charge_time_audit"])
examples, features, targets = prepare_examples(events, traces)
windows = recorded_windows(traces, examples, "randomized")
complete_paths = [row for row in windows if len(row["path"]) == BLOCKS]
reference_paths = audit17["recorded_rollout_totals"]
reference_paths = reference_paths.loc[reference_paths.cohort.eq("randomized")
                                      & reference_paths.horizon_cycles.eq(BLOCK * BLOCKS)]
if ({(row["cell_id"], row["start_cycle"]) for row in complete_paths}
        != set(reference_paths[KEYS].itertuples(index=False, name=None))):
    raise ValueError("The complete recorded windows do not reproduce Notebook 17.")

## Training-only uncertainty margin

For each outer battery fold, fit inner simulators on batteries outside both the outer test fold and the inner validation fold. Replay recorded actions on the inner batteries from measured histories, feeding simulated states back into the forecasts. For each remaining horizon, take the nonnegative 90th percentile of predicted minus measured cumulative Wh. This empirical margin adjusts the energy forecasts used by the controller.

In [3]:
def design_matrices(rows):
    next_rate = rows.charge_rate_c.to_numpy()
    previous = rows.previous_rate_c.to_numpy()
    rate_flags = np.column_stack([(next_rate == rate) for rate in (2, 3)]
                                 + [(previous == rate) for rate in (2, 3)])
    transition_flags = np.column_stack([(previous == a) & (next_rate == b)
                                        for a in RATES for b in RATES if (a, b) != (1, 1)])
    start = rows.start_ah.to_numpy()
    history = rows[["recent_block_delta_ah", "within_block_delta_ah", "age_cycles",
                    "prior_3c_blocks", "prior_2c_blocks"]].to_numpy()
    common = np.column_stack((start, rows.latest_ah, rate_flags, transition_flags, history))
    interactions = start[:, None] * rate_flags
    same = rows[["last_same_rate_ah", "projected_same_rate_ah", "lag_same_rate_cycles"]].to_numpy()
    per_rate = rows[[*(f"last_{rate}c_ah" for rate in RATES),
                     *(f"projected_{rate}c_ah" for rate in RATES),
                     *(f"lag_{rate}c_cycles" for rate in RATES)]].to_numpy()
    return {
        "standard": np.column_stack((common, interactions, history[:, :2] * rate_flags[:, :2])),
        "expanded": np.column_stack((common, same, per_rate, interactions)),
    }


def action_features(history, previous_rate, candidate):
    table = history.copy()
    table["previous_rate_c"] = previous_rate
    table["charge_rate_c"] = candidate
    for feature, suffix in (("last", "ah"), ("projected", "ah"), ("lag", "cycles")):
        history_by_rate = table[[f"{feature}_{rate}c_{suffix}" for rate in RATES]].to_numpy()
        rate_index = np.broadcast_to(np.asarray(candidate, dtype=int), len(table)) - 1
        table[f"{feature}_same_rate_{suffix}"] = history_by_rate[np.arange(len(table)), rate_index]
    return design_matrices(table)


def choose_rate(scores, typical_minutes, target_wh):
    speeds = np.broadcast_to(typical_minutes, scores.shape)
    known = np.isfinite(scores)
    feasible = known & (scores >= target_wh[:, None])
    # Rates are ordered 1C, 2C, 3C, so a tie favors the lower rate.
    quickest = np.where(feasible, speeds, np.inf).argmin(axis=1)
    best_available = np.where(known, scores, -np.inf).argmax(axis=1)
    fallback = ~feasible.any(axis=1)
    return np.asarray(RATES)[np.where(fallback, best_available, quickest)], fallback


def fit_ridge(matrix, labels, cells):
    sample = pd.Series(cells)
    weights = 1 / sample.groupby(sample).transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    usable = np.isfinite(matrix).any(axis=0)
    observed = matrix[:, usable]
    missing = ~np.isfinite(observed)
    indicators = missing.any(axis=0)
    medians = np.nanmedian(np.where(missing, np.nan, observed), axis=0)

    def fill(data):
        absent = ~np.isfinite(data)
        return np.column_stack((np.where(absent, medians, data), absent[:, indicators]))

    filled = fill(observed)
    center = np.average(filled, weights=weights, axis=0)
    scale = np.maximum(np.sqrt(np.average((filled - center)**2, weights=weights, axis=0)), 1e-8)
    fit = Ridge(alpha=1).fit((filled - center) / scale, labels, sample_weight=weights)

    def predict(rows):
        return fit.predict((fill(rows[:, usable]) - center) / scale)
    return predict


def medians_by_rate(train_examples, train_targets):
    values = train_targets.assign(rate=train_examples.charge_rate_c.to_numpy())
    medians = values.groupby("rate")[["ah", "wh", "minutes"]].median().reindex(RATES)
    if not np.isfinite(medians.to_numpy()).all():
        raise ValueError("The training cohort lacks per-rate baseline measurements.")
    return medians.to_numpy()


def fit_simulator(origin, features, targets, include):
    design = action_features(features, features.previous_rate_c, origin.charge_rate_c)
    battery = origin.cell_id.to_numpy()
    specs = {
        "ah": ("standard", "expanded"), "wh": ("standard", "expanded"),
        "first_offset_ah": ("standard",), "last_offset_ah": ("standard",), "minutes": ("standard",),
    }
    training = {}
    for target, families in specs.items():
        values = targets[target].to_numpy()
        usable = include & np.isfinite(values)
        if target == "wh":
            usable &= targets.previous_wh.notna().to_numpy()
        if target == "minutes":
            values = np.where(values > 0, values, np.nan)
            usable &= np.isfinite(values)
            values = np.log(values)
        if not usable.any():
            raise ValueError(f"No training data for {target}.")
        training[target] = {family: fit_ridge(design[family][usable], values[usable], battery[usable])
                            for family in families}

    def predict(rows, candidate):
        features = action_features(rows, rows.previous_rate_c, candidate)
        values = {}
        for target, models in training.items():
            estimate = models["standard"](features["standard"])
            if "expanded" in models:
                estimate = 0.75 * estimate + 0.25 * models["expanded"](features["expanded"])
            values[target] = estimate
        values["ah"] = np.maximum(values["ah"], 0)
        values["wh"] = np.maximum(values["wh"], 0)
        values["minutes"] = np.exp(values["minutes"])
        values["first_ah"] = np.maximum(values["ah"] + values.pop("first_offset_ah"), 0)
        values["last_ah"] = np.maximum(values["ah"] + values.pop("last_offset_ah"), 0)
        return pd.DataFrame(values, index=rows.index)

    return predict, medians_by_rate(origin.loc[include], targets.loc[include])


def latest_measured(history, rate, field, fallback=np.nan):
    return next((getattr(block, field) for block in reversed(history)
                 if block.rate == rate and np.isfinite(getattr(block, field))), fallback)


def baseline_values(history, rate, medians):
    return [latest_measured(history, rate, field, median)
            for (field, median) in zip(("ah", "wh", "minutes"), medians[rate - 1])]


def run_rollouts(scenarios, simulator, medians, policy=None, calibration=None):
    if not scenarios:
        return pd.DataFrame()
    histories = [item["history"] for item in scenarios]
    steps_limit = max(HORIZONS) if policy else max(len(item["path"]) for item in scenarios)
    fixed_rate = {"fixed_1c": 1, "fixed_2c": 2, "fixed_3c": 3}.get(policy)
    outcomes = []
    for step in range(steps_limit):
        active = [i for i, item in enumerate(scenarios) if policy or step < len(item["path"])]
        states = [histories[i] for i in active]
        context = pd.DataFrame(feature_row(history) for history in states)
        target = ENERGY_TARGET * np.asarray([history[-1].wh for history in states])
        fallback = np.zeros(len(states), dtype=bool)
        if policy is None:
            rates = np.asarray([scenarios[i]["path"][step].rate for i in active])
        elif fixed_rate is not None:
            rates = np.full(len(states), fixed_rate)
        else:
            speeds = calibration.training_typical_charge_minutes.to_numpy()
            if policy == "rule":
                scores = np.asarray([[latest_measured(history, rate, "wh") for rate in RATES]
                                     for history in states])
            else:
                forecast = np.column_stack([simulator(context, rate).wh for rate in RATES])
                scores = np.maximum(0, forecast - calibration.overprediction_margin_wh.to_numpy())
            rates, fallback = choose_rate(scores, speeds, target)
        predictions = simulator(context, rates)
        if not np.isfinite(predictions.to_numpy()).all():
            raise ValueError("The simulator returned a nonfinite state.")
        for j, (i, rate, values) in enumerate(zip(active, rates, predictions.itertuples(index=False))):
            scenario = scenarios[i]
            initial = scenario["history"]
            start = scenario["start_cycle"] + step * BLOCK
            batch = Block(start, int(rate), values.ah, values.wh, values.first_ah, values.last_ah, values.minutes)
            histories[i] += (batch,)
            record = dict(cohort=scenario["cohort"], cell_id=scenario["cell_id"], fold=scenario["fold"],
                          start_cycle=scenario["start_cycle"], block_start=start, step=step + 1, rate=int(rate),
                          predicted_ah=values.ah, predicted_wh=values.wh, predicted_minutes=values.minutes,
                          initial_wh=initial[-1].wh, moving_target_wh=target[j], fallback=bool(fallback[j]))
            if policy is None:
                observed = scenario["path"][step]
                baselines = baseline_values(initial, rate, medians)
                record.update(observed_ah=observed.ah, observed_wh=observed.wh, observed_minutes=observed.minutes,
                              baseline_ah=baselines[0], baseline_wh=baselines[1], baseline_minutes=baselines[2])
            else:
                record["policy"] = policy
            outcomes.append(record)
    return pd.DataFrame(outcomes)


def initial_policy_states(traces, examples):
    parts = []
    for cell, group in examples.groupby("cell_id"):
        fold = int(group.fold.iloc[0])
        for start in POLICY_STARTS:
            history = initial_history(traces[cell], start)
            if history is not None:
                parts.append(dict(cohort="randomized", cell_id=cell, start_cycle=start, fold=fold, history=history))
    return parts


def measure_margins(examples, features, targets, windows):
    records = []
    for outer in FOLDS:
        residuals = []
        for inner in FOLDS:
            if inner == outer:
                continue
            fit = ~examples.fold.isin([outer, inner]).to_numpy()
            test = [row for row in windows if row["fold"] == inner]
            if set(examples.cell_id[fit]).intersection(row["cell_id"] for row in test):
                raise ValueError("Inner rollout calibration overlaps training batteries.")
            simulator, medians = fit_simulator(examples, features, targets, fit)
            steps = run_rollouts(test, simulator, medians).sort_values([*KEYS, "step"])
            error = steps.predicted_wh - steps.observed_wh
            steps["cumulative_error_wh"] = error.groupby([steps.cell_id, steps.start_cycle]).cumsum()
            residuals.append(steps[["cell_id", "step", "cumulative_error_wh"]])
        pooled = pd.concat(residuals, ignore_index=True)
        for h, rows in pooled.groupby("step"):
            if rows.empty or not np.isfinite(rows.cumulative_error_wh).all():
                raise ValueError("Incomplete trajectories for margin calibration.")
            margin = max(0, float(rows.cumulative_error_wh.quantile(OVERPREDICTION_QUANTILE)))
            records.append((outer, h, len(rows), rows.cell_id.nunique(), margin))
    return pd.DataFrame(records, columns=["fold", "remaining_blocks", "windows", "batteries", "margin_wh"])


def training_schedule_support(windows):
    records = []
    for outer in FOLDS:
        cells = defaultdict(set)
        for row in windows:
            if row["fold"] != outer:
                sequence = "".join(str(block.rate) for block in row["path"])
                cells[sequence].add(row["cell_id"])
        for sequence in product(RATES, repeat=BLOCKS):
            label = "".join(map(str, sequence))
            records.append((outer, label, len(cells[label])))
    return pd.DataFrame(records, columns=["fold", "sequence", "training_batteries"])


with threadpool_limits(limits=1):
    models = {fold: fit_simulator(examples, features, targets, examples.fold.ne(fold).to_numpy())
              for fold in FOLDS}
    margins = measure_margins(examples, features, targets, windows)
schedule_support = training_schedule_support(complete_paths)
print(f"Calibrated {len(margins)} fold-specific margins from free-running training rollouts.")
print(f"Compared with Notebook 17, {len(complete_paths)} complete 15-cycle recorded windows are available.")

Calibrated 15 fold-specific margins from free-running training rollouts.
Compared with Notebook 17, 801 complete 15-cycle recorded windows are available.


## Energy budget and receding-horizon search

Freeze the budget at 90% of the three-block fixed-1C forecast from the starting state. A schedule qualifies when the Wh already credited plus the conservative forecast for its remaining blocks reaches this budget. Enumerate 27, 9, or 3 schedules according to the number of blocks left. Apply only the first rate of the fastest qualifying schedule. Break ties by higher Wh and then lexicographic rate order. If every schedule falls short, use 1C and log the expected and conservative shortfall of the best 1C-first schedule.

Simulated decisions credit predicted Wh. Recorded replay credits measured Wh only after a selected action matches the logged action.

In [4]:
def forecast_paths(histories, sequences, simulator):
    sequences = np.asarray(sequences, dtype=int)
    roots, plans = len(histories), len(sequences)
    # Evaluate each distinct prefix once per root and carry the next state to its children.
    states = {(): histories}
    wh = {(): np.zeros(roots)}
    minutes = {(): np.zeros(roots)}
    first_blocks = {}
    for depth in range(1, sequences.shape[1] + 1):
        prefixes = sorted({tuple(row[:depth]) for row in sequences})
        history_rows = [history for prefix in prefixes for history in states[prefix[:-1]]]
        rate_rows = np.repeat([prefix[-1] for prefix in prefixes], roots)
        context = pd.DataFrame(feature_row(history) for history in history_rows)
        values = simulator(context, rate_rows)
        if not np.isfinite(values.to_numpy()).all():
            raise ValueError("The simulator returned a nonfinite schedule forecast.")
        aligned = iter(values.itertuples(index=False))
        next_states, next_wh, next_minutes = {}, {}, {}
        for prefix in prefixes:
            prior = states[prefix[:-1]]
            blocks = [Block(history[-1].start + BLOCK, prefix[-1], row.ah, row.wh,
                            row.first_ah, row.last_ah, row.minutes)
                      for history, row in zip(prior, islice(aligned, roots), strict=True)]
            next_states[prefix] = [history + (block,) for history, block in zip(prior, blocks)]
            next_wh[prefix] = wh[prefix[:-1]] + np.array([block.wh for block in blocks])
            next_minutes[prefix] = minutes[prefix[:-1]] + np.array([block.minutes for block in blocks])
            if depth == 1:
                first_blocks[prefix[-1]] = blocks
        states, wh, minutes = next_states, next_wh, next_minutes
    assert len(wh) == plans
    return (np.column_stack([wh[tuple(row)] for row in sequences]),
            np.column_stack([minutes[tuple(row)] for row in sequences]), first_blocks)


def preferred_paths(sequences, predicted_wh, minutes, required_wh, margin):
    conservative = np.maximum(0, predicted_wh - margin)
    feasible = conservative + TOLERANCE >= required_wh[:, None]
    first_is_1c = sequences[:, 0] == 1
    selected = []
    for index, valid in enumerate(feasible):
        if valid.any():
            possible = np.flatnonzero(valid)
            choice = possible[np.lexsort((possible, -predicted_wh[index, possible], minutes[index, possible]))[0]]
        else:
            possible = np.flatnonzero(first_is_1c)
            choice = possible[np.lexsort((possible, minutes[index, possible], -predicted_wh[index, possible]))[0]]
        selected.append(choice)
    return np.asarray(selected), ~feasible.any(axis=1), conservative


def next_budget_action(histories, simulator, remaining, budget, credited, margin):
    sequences = np.asarray(list(product(RATES, repeat=remaining)))
    forecast, minutes, first_blocks = forecast_paths(histories, sequences, simulator)
    required = budget - credited
    selected, fallback, conservative = preferred_paths(sequences, forecast, minutes, required, margin)
    decisions = []
    for index, choice in enumerate(selected):
        sequence = sequences[choice]
        estimate = float(forecast[index, choice])
        lower = float(conservative[index, choice])
        decisions.append(dict(block=first_blocks[sequence[0]][index],
                              planned_sequence="".join(map(str, sequence)),
                              remaining_forecast_wh=estimate, remaining_lower_wh=lower,
                              planned_remaining_minutes=float(minutes[index, choice]),
                              remaining_budget_wh=float(required[index]),
                              fallback=bool(fallback[index]),
                              expected_shortfall_wh=max(0, float(required[index]) - estimate),
                              conservative_shortfall_wh=max(0, float(required[index]) - lower)))
    return decisions


def budget_for_start(histories, simulator):
    fixed_path = [(1,) * BLOCKS]
    fixed_wh, _, _ = forecast_paths(histories, fixed_path, simulator)
    return fixed_wh[:, 0], ENERGY_TARGET * fixed_wh[:, 0]


def select_context(scenario, step):
    return dict(cell_id=scenario["cell_id"], fold=scenario["fold"], start_cycle=scenario["start_cycle"],
                block_start=scenario["start_cycle"] + step * BLOCK, step=step + 1)


def simulate_budget(scenarios, simulator, margin):
    histories = [row["history"] for row in scenarios]
    reference, budget = budget_for_start(histories, simulator)
    credit = np.zeros(len(scenarios))
    trace = []
    for step in range(BLOCKS):
        remaining = BLOCKS - step
        for i, decision in enumerate(next_budget_action(histories, simulator, remaining, budget, credit, margin[remaining])):
            block = decision.pop("block")
            record = select_context(scenarios[i], step)
            record.update(policy="energy_budget", rate=block.rate, predicted_wh=block.wh, predicted_minutes=block.minutes,
                          predicted_ah=block.ah, budget_wh=budget[i], fixed_1c_forecast_wh=reference[i],
                          credited_wh_before=credit[i], credited_wh_after=credit[i] + block.wh,
                          initial_wh=scenarios[i]["history"][-1].wh, margin_wh=margin[remaining], **decision)
            trace.append(record)
            histories[i] += (block,)
            credit[i] += block.wh
    return pd.DataFrame(trace)


def verify_selection():
    options = np.asarray([(1, 1), (1, 2), (2, 1), (3, 1)])
    energy = np.array([[25, 24, 22, 19], [25, 24, 22, 19], [25, 24, 22, 19]], dtype=float)
    time = np.array([[100, 75, 60, 50]] * len(energy), dtype=float)
    picked, fallback, _ = preferred_paths(options, energy, time, np.array([21, 23, 40]), margin=1)
    np.testing.assert_array_equal(picked, [2, 1, 0])
    np.testing.assert_array_equal(fallback, [False, False, True])


verify_selection()

In [5]:
def run_simulation(models, margins, starting_states, calibration):
    outputs = []
    for fold, (simulator, medians) in models.items():
        states = [row for row in starting_states if row["fold"] == fold]
        original = calibration.loc[calibration.fold.eq(fold)].set_index("candidate_rate_c").reindex(RATES)
        guard = margins.loc[margins.fold.eq(fold)].set_index("remaining_blocks").margin_wh
        fixed = run_rollouts(states, simulator, medians, "fixed_1c", original)
        moving = run_rollouts(states, simulator, medians, "model", original)
        adaptive = simulate_budget(states, simulator, guard)
        anchors = adaptive.loc[adaptive.step.eq(1), [*KEYS, "fixed_1c_forecast_wh", "budget_wh"]]
        for baseline in (fixed, moving):
            baseline = baseline.merge(anchors, on=KEYS, validate="many_to_one")
            baseline = baseline.sort_values([*KEYS, "step"])
            baseline["credited_wh_after"] = baseline.groupby(KEYS).predicted_wh.cumsum()
            baseline["credited_wh_before"] = baseline["credited_wh_after"] - baseline.predicted_wh
            outputs.append(baseline)
        outputs.append(adaptive)
    return pd.concat(outputs, ignore_index=True)


def verify_simulation(steps, reference17):
    keys = [*KEYS, "fold", "policy", "step", "block_start"]
    baseline = steps.loc[steps.policy.isin(["fixed_1c", "model"])].set_index(keys, verify_integrity=True).sort_index()
    reference = reference17.loc[reference17.policy.isin(["fixed_1c", "model"]) & reference17.step.le(BLOCKS)]
    reference = reference.set_index(keys, verify_integrity=True).sort_index()
    pd.testing.assert_index_equal(baseline.index, reference.index)
    for column in ("rate", "fallback"):
        np.testing.assert_array_equal(baseline[column], reference[column])
    for column in ("predicted_wh", "predicted_ah", "predicted_minutes"):
        np.testing.assert_allclose(baseline[column], reference[column], atol=TOLERANCE, rtol=0)

    adaptive = steps.loc[steps.policy.eq("energy_budget")].sort_values([*KEYS, "step"])
    groups = adaptive.groupby(KEYS)
    np.testing.assert_allclose(groups.budget_wh.max(), groups.budget_wh.min(), atol=TOLERANCE, rtol=0)
    np.testing.assert_allclose(groups.credited_wh_after.last(), groups.predicted_wh.sum(), atol=TOLERANCE, rtol=0)
    assert groups.size().eq(BLOCKS).all()
    np.testing.assert_allclose(adaptive.budget_wh, ENERGY_TARGET * adaptive.fixed_1c_forecast_wh, atol=TOLERANCE, rtol=0)
    np.testing.assert_allclose(adaptive.remaining_budget_wh, adaptive.budget_wh - adaptive.credited_wh_before,
                               atol=TOLERANCE, rtol=0)
    np.testing.assert_allclose(adaptive.credited_wh_before,
                               groups.credited_wh_after.shift(fill_value=0), atol=TOLERANCE, rtol=0)
    np.testing.assert_array_equal(adaptive.rate.astype(str), adaptive.planned_sequence.str[0])
    assert adaptive.loc[adaptive.fallback, "rate"].eq(1).all()
    assert adaptive.loc[~adaptive.fallback, "conservative_shortfall_wh"].le(TOLERANCE).all()
    fixed_only = baseline.reset_index().loc[lambda frame: frame.policy.eq("fixed_1c")]
    final_fixed = fixed_only.groupby(KEYS).predicted_wh.sum()
    np.testing.assert_allclose(groups.fixed_1c_forecast_wh.first(), final_fixed, atol=TOLERANCE, rtol=0)


with threadpool_limits(limits=1):
    starting_states = initial_policy_states(traces, examples)
    simulated = run_simulation(models, margins, starting_states, audit16["calibration"])
    verify_simulation(simulated, audit17["policy_trajectories"])
print(f"Simulated {len(starting_states)} shared starting states and reproduced both Notebook 17 baselines.")

Simulated 164 shared starting states and reproduced both Notebook 17 baselines.


## Randomized replay

Include every eligible recorded starting state with measured history, three verified logged rates, and sufficient scheduled follow-up. Do not require future Wh or time to include an origin. A policy first selects a rate from the available state. Compare it with the logged rate, then read the measured outcome only after a match. Stop on an action mismatch or missing feedback needed for a subsequent choice. Count full three-action matches separately from matches with complete measured outcomes.

This is a separate check on actual measurements. The randomized logger chose each rate with probability 1/3, giving full matches an inverse propensity of 27. The replay normalizes weights within each outcome and gives each battery equal total weight before matching. Exact-match estimates can be uncertain when few batteries contribute.

In [6]:
def replay_origins(examples, traces):
    logged = examples.set_index(KEYS, verify_integrity=True).charge_rate_c.to_dict()
    upper = LAST_START - BLOCK * (BLOCKS - 1)
    states = []
    for cell, start, fold in examples[[*KEYS, "fold"]].itertuples(index=False, name=None):
        if not FIRST_START <= start <= upper:
            continue
        history = initial_history(traces[cell], start)
        if history is None:
            continue
        cycles = [start + offset * BLOCK for offset in range(BLOCKS)]
        rates = tuple(logged.get((cell, cycle)) for cycle in cycles)
        if all(rate in RATES for rate in rates):
            states.append(dict(cell_id=cell, start_cycle=start, fold=fold, history=history, logged_rates=rates))
    return states


def choose_replay_actions(states, policy, simulator, calibration, remaining, margin):
    if policy == "fixed_1c":
        return [dict(rate=1, fallback=False) for _ in states]
    histories = [row["history"] for row in states]
    if policy == "energy_budget":
        budget = np.asarray([row["budget_wh"] for row in states])
        credited = np.asarray([sum(row["measured_wh"]) for row in states])
        choices = next_budget_action(histories, simulator, remaining, budget, credited, margin)
        for row in choices:
            row["rate"] = row.pop("block").rate
        return choices
    context = pd.DataFrame(feature_row(history) for history in histories)
    target = ENERGY_TARGET * np.asarray([history[-1].wh for history in histories])
    forecasts = np.column_stack([simulator(context, rate).wh for rate in RATES])
    conservative = np.maximum(0, forecasts - calibration.overprediction_margin_wh.to_numpy())
    rates, fallback = choose_rate(conservative, calibration.training_typical_charge_minutes.to_numpy(), target)
    return [dict(rate=int(rate), fallback=bool(flag)) for rate, flag in zip(rates, fallback)]


def replay_policy(scenarios, policy, simulator, calibration, guard, traces):
    _, budgets = budget_for_start([row["history"] for row in scenarios], simulator)
    states = [{**row, "budget_wh": float(budget), "matched_steps": 0, "termination": "completed",
               "measured_wh": [], "measured_minutes": []} for row, budget in zip(scenarios, budgets)]
    active = list(states)
    ledger = []
    for step in range(BLOCKS):
        if not active:
            break
        remaining = BLOCKS - step
        decisions = choose_replay_actions(active, policy, simulator, calibration, remaining, guard[remaining])
        continued = []
        for state, decision in zip(active, decisions):
            log_rate = state["logged_rates"][step]
            context = select_context(state, step)
            decision_made = dict(**context, policy=policy, logged_rate=log_rate, budget_wh=state["budget_wh"],
                                 matched=bool(decision["rate"] == log_rate), measured_wh=np.nan,
                                 measured_minutes=np.nan, **decision)
            ledger.append(decision_made)
            if not decision_made["matched"]:
                state["termination"] = "action_mismatch"
                continue

            # Only matching actions reveal the outcome of the policy's selected action.
            observed = traces[state["cell_id"]].get(context["block_start"])
            state["matched_steps"] += 1
            wh = observed.wh if observed is not None else np.nan
            minutes = observed.minutes if observed is not None else np.nan
            state["measured_wh"].append(wh)
            state["measured_minutes"].append(minutes)
            decision_made.update(measured_wh=wh, measured_minutes=minutes)
            if step + 1 < BLOCKS and policy != "fixed_1c":
                if observed is None or not np.isfinite(wh):
                    state["termination"] = "missing_feedback"
                    continue
                state["history"] += (observed,)
            continued.append(state)
        active = continued

    matches = []
    for state in states:
        matched = state["matched_steps"] == BLOCKS
        matches.append(dict(cell_id=state["cell_id"], start_cycle=state["start_cycle"], fold=state["fold"],
                            policy=policy, matched_steps=state["matched_steps"], full_match=matched,
                            termination=state["termination"], budget_wh=state["budget_wh"],
                            measured_wh=float(np.sum(state["measured_wh"])) if matched else np.nan,
                            measured_minutes=float(np.sum(state["measured_minutes"])) if matched else np.nan))
    return pd.DataFrame(matches), pd.DataFrame(ledger)


def aggregate_replay(replay):
    results = []
    for policy, frame in replay.groupby("policy"):
        per_battery = frame.groupby("cell_id").start_cycle.transform("size")
        inverse_propensity = len(RATES)**BLOCKS
        weights = inverse_propensity / per_battery
        matched = frame.full_match
        wh = matched & frame.measured_wh.notna()
        time = matched & frame.measured_minutes.notna()
        outcome = matched & wh & time
        stats = dict(policy=policy, origins=len(frame), batteries=frame.cell_id.nunique(), full_matches=int(matched.sum()),
                     matched_batteries=frame.loc[matched, "cell_id"].nunique(),
                     complete_outcomes=int(outcome.sum()), complete_batteries=frame.loc[outcome, "cell_id"].nunique(),
                     energy_matches=int(wh.sum()), energy_batteries=frame.loc[wh, "cell_id"].nunique(),
                     time_matches=int(time.sum()), time_batteries=frame.loc[time, "cell_id"].nunique(),
                     missing_feedback=int(frame.termination.eq("missing_feedback").sum()))
        shortfalls = (frame.loc[wh, "budget_wh"] - frame.loc[wh, "measured_wh"]).clip(lower=0)
        stats["measured_budget_hits"] = int(shortfalls.le(TOLERANCE).sum())
        stats["measured_worst_shortfall_wh"] = shortfalls.max()
        stats["measured_wh"] = np.average(frame.loc[wh, "measured_wh"], weights=weights[wh]) if wh.any() else np.nan
        stats["measured_minutes"] = (
            np.average(frame.loc[time, "measured_minutes"], weights=weights[time]) if time.any() else np.nan
        )
        results.append(stats)
    return pd.DataFrame(results).set_index("policy").reindex(POLICIES).reset_index()


def run_recorded_replay(models, margins, calibration, examples, traces):
    scenarios = replay_origins(examples, traces)
    paths, steps = [], []
    for fold, (simulator, _) in models.items():
        states = [row for row in scenarios if row["fold"] == fold]
        original = calibration.loc[calibration.fold.eq(fold)].set_index("candidate_rate_c").reindex(RATES)
        guard = margins.loc[margins.fold.eq(fold)].set_index("remaining_blocks").margin_wh
        for policy in POLICIES:
            path, trace = replay_policy(states, policy, simulator, original, guard, traces)
            paths.append(path)
            steps.append(trace)
    outcomes = pd.concat(paths, ignore_index=True)
    ledger = pd.concat(steps, ignore_index=True)
    require_unique(outcomes, [*KEYS, "policy"], "replay paths")
    require_unique(ledger, [*KEYS, "step", "policy"], "replay decisions")
    assert outcomes.groupby("policy").size().eq(len(scenarios)).all()
    assert ledger.loc[~ledger.matched, ["measured_wh", "measured_minutes"]].isna().all().all()
    return outcomes, ledger


def verify_replay_isolation(models, margins, calibration, scenarios, traces):
    scenario = scenarios[0]
    fold = scenario["fold"]
    simulator, _ = models[fold]
    original = calibration.loc[calibration.fold.eq(fold)].set_index("candidate_rate_c").reindex(RATES)
    guard = margins.loc[margins.fold.eq(fold)].set_index("remaining_blocks").margin_wh
    _, first = replay_policy([scenario], "energy_budget", simulator, original, guard, traces)
    # Alter a first future measurement while preserving its logged action.
    cell, cycle = scenario["cell_id"], scenario["start_cycle"]
    changed = traces[cell].get(cycle)
    assert changed is not None
    replacement = Block(cycle, changed.rate, 0.25, 3.0, 0.25, 0.25, changed.minutes)
    masked = {**traces, cell: {**traces[cell], cycle: replacement}}
    _, second = replay_policy([scenario], "energy_budget", simulator, original, guard, masked)
    columns = ["rate", "planned_sequence", "budget_wh", "remaining_forecast_wh", "fallback"]
    pd.testing.assert_series_equal(first.loc[0, columns], second.loc[0, columns])


with threadpool_limits(limits=1):
    replay_paths, replay_steps = run_recorded_replay(models, margins, audit16["calibration"], examples, traces)
    verify_replay_isolation(models, margins, audit16["calibration"], replay_origins(examples, traces), traces)
replay_metrics = aggregate_replay(replay_paths)
print(replay_metrics[["policy", "origins", "full_matches", "matched_batteries", "complete_outcomes"]].to_string(index=False))

       policy  origins  full_matches  matched_batteries  complete_outcomes
     fixed_1c      819            21                 19                 19
        model      819            46                 28                 45
energy_budget      819            24                 18                 24


## Evaluation criteria

The simulation advances only when the upper endpoint of the paired 95% interval for charging-time change is below zero and the lower endpoint for cumulative Wh minus 90% of fixed 1C's Wh is at least zero. Resample whole batteries with shared draws across both outcomes. Average starts within each battery before averaging batteries. Budget attainment and shortfalls describe individual starts separately from the mean criterion.

In [7]:
def bootstrap_cell_means(cells):
    data = cells.to_numpy(dtype=float)
    if len(data) < 2:
        return np.full((2, data.shape[1]), np.nan)
    rng = np.random.default_rng(SEED)
    indices = rng.integers(0, len(data), size=(BOOTSTRAP_DRAWS, len(data)))
    return np.quantile(data[indices].mean(axis=1), [0.025, 0.975], axis=0)


def score_simulation(simulated):
    totals = (simulated.sort_values("step").groupby([*KEYS, "fold", "policy"], as_index=False)
              .agg(wh=("predicted_wh", "sum"), minutes=("predicted_minutes", "sum"),
                   budget_wh=("budget_wh", "first"), blocks=("step", "size"),
                   fallback_blocks=("fallback", "sum"), sequence=("rate", lambda values: "".join(map(str, values)))))
    assert totals.blocks.eq(BLOCKS).all()
    totals["shortfall_wh"] = np.maximum(0, totals.budget_wh - totals.wh)
    totals["budget_met"] = totals.shortfall_wh.le(TOLERANCE)
    fixed = totals.loc[totals.policy.eq("fixed_1c"), [*KEYS, "wh", "minutes"]]
    totals = totals.merge(fixed, on=KEYS, validate="many_to_one", suffixes=("", "_fixed_1c"))
    totals["delta_minutes"] = totals.minutes - totals.minutes_fixed_1c
    totals["delta_wh"] = totals.wh - totals.wh_fixed_1c
    totals["energy_margin_wh"] = totals.wh - ENERGY_TARGET * totals.wh_fixed_1c

    records = []
    columns = ["wh", "minutes", "delta_minutes", "delta_wh", "wh_fixed_1c", "energy_margin_wh"]
    for policy, rows in totals.groupby("policy"):
        batteries = rows.groupby("cell_id")[columns].mean()
        means = batteries.mean()
        ci_low, ci_high = bootstrap_cell_means(batteries[["delta_minutes", "energy_margin_wh"]])
        worst = rows.sort_values(["shortfall_wh", "cell_id", "start_cycle"],
                                 ascending=[False, True, True]).iloc[0]
        n_worst = int(np.ceil(0.1 * len(rows)))
        records.append(dict(policy=policy, origins=len(rows), batteries=len(batteries), **means.to_dict(),
                            energy_fraction=means.wh / means.wh_fixed_1c,
                            budget_hits=int(rows.budget_met.sum()), fallback_starts=int(rows.fallback_blocks.gt(0).sum()),
                            fallback_blocks=int(rows.fallback_blocks.sum()), total_blocks=int(rows.blocks.sum()),
                            worst_decile_shortfall_wh=float(rows.shortfall_wh.nlargest(n_worst).mean()),
                            worst_shortfall_wh=float(worst.shortfall_wh),
                            worst_cell=int(worst.cell_id) if worst.shortfall_wh > TOLERANCE else np.nan,
                            worst_start_cycle=int(worst.start_cycle) if worst.shortfall_wh > TOLERANCE else np.nan,
                            delta_minutes_ci95_low=ci_low[0], delta_minutes_ci95_high=ci_high[0],
                            energy_margin_ci95_low=ci_low[1], energy_margin_ci95_high=ci_high[1]))
    summary = pd.DataFrame(records).set_index("policy").reindex(POLICIES).reset_index()
    candidate = summary.set_index("policy").loc["energy_budget"]
    time_pass = bool(candidate.delta_minutes_ci95_high < 0)
    energy_pass = bool(candidate.energy_margin_ci95_low >= 0)
    acceptance = pd.DataFrame([dict(time_pass=time_pass, energy_pass=energy_pass,
                                    qualified_in_simulation=time_pass and energy_pass)])
    return totals, summary, acceptance


with threadpool_limits(limits=1):
    totals, metrics, acceptance = score_simulation(simulated)
print(metrics[["policy", "minutes", "wh", "energy_fraction", "budget_hits", "fallback_blocks"]].round(3).to_string(index=False))
print(acceptance.to_string(index=False))

       policy  minutes     wh  energy_fraction  budget_hits  fallback_blocks
     fixed_1c  603.667 81.850            1.000          164                0
        model  397.263 70.631            0.863           66                0
energy_budget  483.782 76.183            0.931          164                0
 time_pass  energy_pass  qualified_in_simulation
      True         True                     True


In [8]:
def report_text(metrics, acceptance, replay, margins, schedules):
    candidate = metrics.set_index("policy").loc["energy_budget"]
    moving = metrics.set_index("policy").loc["model"]
    fixed = metrics.set_index("policy").loc["fixed_1c"]
    outcome = acceptance.iloc[0]
    gate = "met both simulation criteria" if outcome.qualified_in_simulation else "fell short of the simulation criteria"
    lines = [
        "# MIT–BIT fixed energy budget controller", "",
        "## Results", "",
        f"**The fixed-budget controller {gate}.** Over 15 cycles it delivered "
        f"{candidate.energy_fraction:.1%} of fixed 1C's simulated Wh, compared with "
        f"{moving.energy_fraction:.1%} for the moving-target controller. It changed mean charging time "
        f"by {candidate.delta_minutes:+.1f} minutes ({candidate.delta_minutes / fixed.minutes:+.1%}) "
        f"relative to fixed 1C.", "",
        f"The simulation compared all three policies on the same {int(candidate.origins)} starts across "
        f"{int(candidate.batteries)} held-out batteries. Each five-cycle action used 1C, 2C, or 3C "
        f"charging with 3C discharge throughout.", "",
        "| Policy | Charge time (min) | Delivered energy (Wh) | Energy vs fixed 1C | Budget hits | Fallback blocks |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ]
    for row in metrics.itertuples(index=False):
        lines.append(f"| {POLICIES[row.policy]} | {row.minutes:.1f} | {row.wh:.2f} | "
                     f"{row.energy_fraction:.1%} | {int(row.budget_hits)}/{int(row.origins)} | "
                     f"{int(row.fallback_blocks)}/{int(row.total_blocks)} |")
    lines.extend(["", "Charging time and energy give each battery equal weight. Budget hits count individual starts. "
                  "Fallbacks use the rule defined by each policy.", "",
                  "| Policy | Average shortfall in worst 10% (Wh) | Maximum shortfall (Wh) | Worst start |",
                  "| --- | ---: | ---: | --- |"])
    for row in metrics.itertuples(index=False):
        location = f"Cell #{int(row.worst_cell)} at cycle {int(row.worst_start_cycle)}" if np.isfinite(row.worst_cell) else "—"
        lines.append(f"| {POLICIES[row.policy]} | {row.worst_decile_shortfall_wh:.2f} | "
                     f"{row.worst_shortfall_wh:.2f} | {location} |")
    time = "Pass" if outcome.time_pass else "Fail"
    energy = "Pass" if outcome.energy_pass else "Fail"
    lines.extend([
        "", "## Paired decision criteria", "",
        "| Criterion | Difference | 95% battery bootstrap interval | Result |",
        "| --- | ---: | ---: | --- |",
        f"| Charging time relative to fixed 1C | {candidate.delta_minutes:+.1f} min | "
        f"{candidate.delta_minutes_ci95_low:+.1f} to {candidate.delta_minutes_ci95_high:+.1f} min | {time} |",
        f"| Delivered Wh relative to 90% of fixed 1C | {candidate.energy_margin_wh:+.2f} Wh | "
        f"{candidate.energy_margin_ci95_low:+.2f} to {candidate.energy_margin_ci95_high:+.2f} Wh | {energy} |",
        "",
        f"The controller fixed the target at {ENERGY_TARGET:.0%} of the initial 15-cycle fixed-1C forecast. "
        "It replanned after each block and credited the energy delivered before the next decision. "
        f"Margins used the {OVERPREDICTION_QUANTILE:.0%} quantile of overprediction from inner held-out training "
        f"rollouts. The minimum sequence support was {int(schedules.training_batteries.min())} training batteries "
        "across all 27 three-block schedules and all five outer folds.", "",
        "| Remaining blocks | Smallest fold margin (Wh) | Largest fold margin (Wh) |",
        "| ---: | ---: | ---: |",
    ])
    for remaining, rows in margins.groupby("remaining_blocks"):
        lines.append(f"| {remaining} | {rows.margin_wh.min():.2f} | {rows.margin_wh.max():.2f} |")
    lines.extend([
        "", "## Measured randomized replay", "",
        f"Replay considered {int(replay.origins.iloc[0])} eligible starting points across "
        f"{int(replay.batteries.iloc[0])} batteries. At each exact match, the next decision received the measured "
        "outcome. A full match requires the recorded action to agree with the policy at all three steps.", "",
        "| Policy | Full matches (batteries) | Complete Wh and time (batteries) | Measured time (min) | "
        "Measured energy (Wh) | Measured budget hits |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ])
    for row in replay.itertuples(index=False):
        lines.append(f"| {POLICIES[row.policy]} | {int(row.full_matches)} ({int(row.matched_batteries)}) | "
                     f"{int(row.complete_outcomes)} ({int(row.complete_batteries)}) | "
                     f"{row.measured_minutes:.1f} | {row.measured_wh:.2f} | {int(row.measured_budget_hits)}/{int(row.energy_matches)} |")
    lines.extend([
        "",
        "The replay estimates use outcome-specific availability, normalized inverse-propensity weights, and "
        "equal battery weight before matching. Each policy has its own set of matching trajectories. "
        "These estimates provide a descriptive check on the simulation and have limited support for a policy-effect claim. "
        "Planning and scoring within the simulation use the same transition model. The replay uses measured outcomes.",
        "",
        "The analysis addresses 15-cycle operation. A supported 30-cycle comparison needs measured sustained 1C "
        "trajectories. Battery lifetime requires a separate experiment.", "",
        "[MIT–BIT v3 dataset and experimental design](https://data.mendeley.com/datasets/kw34hhw7xg/3)", "",
    ])
    return "\n".join(lines)


def plot_examples(steps, totals, path):
    import matplotlib.pyplot as plt

    fixed_budget = totals.loc[totals.policy.eq("energy_budget")]
    # Show a representative changed schedule and the largest shortfall from the previous policy.
    original = totals.loc[totals.policy.eq("model"), [*KEYS, "sequence", "shortfall_wh"]]
    comparison = fixed_budget.merge(original, on=KEYS, suffixes=("", "_moving"), validate="one_to_one")
    distinct = comparison.loc[comparison.sequence.ne(comparison.sequence_moving)
                              & comparison.sequence.ne("111") & comparison.sequence_moving.ne("111")]
    pool = distinct if not distinct.empty else comparison
    central = (pool.delta_minutes - pool.delta_minutes.median()).abs().idxmin()
    worst = comparison.shortfall_wh_moving.idxmax()
    examples = [(pool.loc[central], "Representative charging-time change"),
                (comparison.loc[worst], "Largest moving-target shortfall")]
    palette = {"fixed_1c": "#416b97", "model": "#b65333", "energy_budget": "#187c64"}
    fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout="constrained")
    for (example, caption), (energy, time) in zip(examples, axes):
        selection = steps.loc[steps.cell_id.eq(example.cell_id) & steps.start_cycle.eq(example.start_cycle)]
        for policy, label in POLICIES.items():
            rows = selection.loc[selection.policy.eq(policy)].sort_values("step")
            cycles = BLOCK * rows.step
            energy.plot(cycles, rows.credited_wh_after, "-o", color=palette[policy], label=label)
            time.plot(cycles, rows.predicted_minutes.cumsum(), "-o", color=palette[policy], label=label)
        energy.scatter(BLOCKS * BLOCK, example.budget_wh, facecolor="white", edgecolor="#333333",
                       marker="D", zorder=4, label="15-cycle budget")
        energy.set_title(f"{caption}\nCell #{int(example.cell_id)} at cycle {int(example.start_cycle)}")
        energy.set_ylabel("Cumulative energy (Wh)")
        time.set_title("Charging time")
        time.set_ylabel("Cumulative charging time (min)")
        for ax in (energy, time):
            ax.set_xticks([BLOCK * h for h in HORIZONS])
            ax.set_xlabel("Cycles after start")
            ax.grid(alpha=0.2)
    axes[0, 0].legend(fontsize=8)
    fig.savefig(path, dpi=145)
    plt.close(fig)


output.mkdir(parents=True, exist_ok=True)
tables = {
    "simulation_metrics": metrics, "simulation_totals": totals, "simulation_trajectories": simulated,
    "acceptance": acceptance, "calibration_margins": margins, "training_sequence_support": schedule_support,
    "replay_metrics": replay_metrics, "replay_paths": replay_paths, "replay_steps": replay_steps,
}
for name, table in tables.items():
    table.to_csv(output / f"{name}.csv", index=False)
config = dict(dataset_doi="10.17632/kw34hhw7xg.3", horizon_cycles=BLOCK * BLOCKS, block_size=BLOCK,
              policy_starts=POLICY_STARTS, budget_fraction=ENERGY_TARGET,
              overprediction_quantile=OVERPREDICTION_QUANTILE, folds=FOLDS, bootstrap_draws=BOOTSTRAP_DRAWS,
              bootstrap_seed=SEED, per_action_log_probability=1 / len(RATES))
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(report_text(metrics, acceptance, replay_metrics, margins, schedule_support),
                                   encoding="utf-8")
plot_examples(simulated, totals, output / "example_trajectories.png")
files = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "example_trajectories.png"]
archive_path = output.with_suffix(".zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as bundle:
    for name in files:
        bundle.write(output / name, arcname=f"{output.name}/{name}")
print(f"Report: {output / 'summary.md'}")
print(f"Download: {archive_path}")

Report: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_energy_budget_controller_results/summary.md
Download: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_energy_budget_controller_results.zip
